# Advanced Professional Baseline Notebook Structure

## ROGII - Wellbore Geology Prediction

---

# 1. Notebook Header

```python
# ============================================================
# ROGII - Wellbore Geology Prediction
# Advanced Baseline Pipeline
#
# Author: Md Ashraf
# IIT (ISM) Dhanbad
# ============================================================
```

---

In [1]:

from IPython.core import getipython
from IPython.core import getipython
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 200)

# =============================
# PATHS
# =============================

ROOT_DIR = Path("/media/ashraf/Windows/Users/MD ASHRAF/Documents/wellbore-geology-prediction-Well-log/data/raw")

TRAIN_DIR = ROOT_DIR / "train"
TEST_DIR = ROOT_DIR / "test"

SUBMISSION_PATH = ROOT_DIR / "sample_submission.csv"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# 4. Competition Overview
print("="*60)
print("ROGII - Wellbore Geology Prediction")
print("Target: Predict TVT along horizontal wells")
print("Metric: RMSE")
print("="*60)




ROGII - Wellbore Geology Prediction
Target: Predict TVT along horizontal wells
Metric: RMSE


In [2]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def rmse(y_true, y_pred):
    """Calculate Root Mean Squared Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

def reduce_mem_usage(df):
    """Iterate through all columns of a dataframe and modify the data type to reduce memory usage."""
    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)

    return df

## 📂 Data Loading
Extracting horizontal well data from individual files, appending identifiers, and concatenating them into our primary dataframes.

In [3]:
# ============================================================
# 4. DATA LOADING
# ============================================================

# --- Train Data ---
train_files = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_data = []

for file in tqdm(train_files, desc="Loading Train Files"):
    well_name = file.stem.split("__")[0]
    df = pd.read_csv(file)
    df["WELL"] = well_name
    df["ROW_IDX"] = np.arange(len(df))
    train_data.append(df)

train_df = pd.concat(train_data, ignore_index=True)
train_df = reduce_mem_usage(train_df)
print(f"Train Shape: {train_df.shape}")

# --- Test Data ---
test_files = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
test_data = []

for file in tqdm(test_files, desc="Loading Test Files"):
    well_name = file.stem.split("__")[0]
    df = pd.read_csv(file)
    df["WELL"] = well_name
    df["ROW_IDX"] = np.arange(len(df))
    test_data.append(df)

test_df = pd.concat(test_data, ignore_index=True)
test_df = reduce_mem_usage(test_df)
print(f"Test Shape: {test_df.shape}")

Loading Train Files:   0%|          | 0/773 [00:00<?, ?it/s]

Train Shape: (5092255, 15)


Loading Test Files:   0%|          | 0/3 [00:00<?, ?it/s]

Test Shape: (19221, 8)


In [5]:
print("Train data columns",train_df.columns)
print("Test data columns",test_df.columns)

Train data columns Index(['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA',
       'TVT', 'GR', 'TVT_input', 'WELL', 'ROW_IDX'],
      dtype='object')
Test data columns Index(['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input', 'WELL', 'ROW_IDX'], dtype='object')


## 📊 Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# 5. EXPLORATORY DATA ANALYSIS
# ============================================================

# Missing Values
missing = train_df.isnull().mean().sort_values(ascending=False)
display(missing[missing > 0])

# Target Distribution
fig = px.histogram(
    train_df,
    x="TVT",
    nbins=100,
    title="TVT Distribution",
    color_discrete_sequence=['#3498db']
)
fig.show()

# Well-wise Visualization
sample_well = train_df["WELL"].unique()[0]
well_df = train_df[train_df["WELL"] == sample_well]

fig = px.line(
    well_df,
    x="MD",
    y=["GR", "TVT"],
    title=f"Well Analysis: {sample_well}"
)
fig.show()

TVT_input    0.743087
GR           0.296130
ANCC         0.008961
EGFDL        0.001191
dtype: float64

In [6]:
# ============================================================
# GEOLOGICAL RELATIVE POSITION FEATURES
# ============================================================

surface_cols = [
    "ANCC",
    "ASTNU",
    "ASTNL",
    "EGFDU",
    "EGFDL",
    "BUDA"
]

for col in surface_cols:

    # Train
    if col in train_df.columns:
        train_df[f"Z_minus_{col}"] = train_df["Z"] - train_df[col]

    # Test
    if col in test_df.columns:
        test_df[f"Z_minus_{col}"] = test_df["Z"] - test_df[col]

In [ ]:
# ============================================================
# ROGII - Wellbore Geology Prediction
# Advanced Professional Baseline
#
# Author: Md Ashraf
# IIT (ISM) Dhanbad
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ============================================================
# PATHS
# ============================================================

ROOT_DIR = Path(
    "/media/ashraf/Windows/Users/MD ASHRAF/Documents/wellbore-geology-prediction-Well-log/data/raw"
)

TRAIN_DIR = ROOT_DIR / "train"
TEST_DIR = ROOT_DIR / "test"

SUBMISSION_PATH = ROOT_DIR / "sample_submission.csv"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ============================================================
# OVERVIEW
# ============================================================

print("=" * 60)
print("ROGII - Wellbore Geology Prediction")
print("Target: Predict TVT along horizontal wells")
print("Metric: RMSE")
print("=" * 60)

# ============================================================
# METRIC
# ============================================================

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ============================================================
# LOAD TRAIN DATA
# ============================================================

print("\nLoading TRAIN data...")

train_files = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))

train_data = []

for file in tqdm(train_files):

    well_name = file.stem.split("__")[0]

    df = pd.read_csv(file)

    df["WELL"] = well_name
    df["ROW_IDX"] = np.arange(len(df))

    train_data.append(df)

train_df = pd.concat(train_data, ignore_index=True)

print(f"Train Shape: {train_df.shape}")

# ============================================================
# LOAD TEST DATA
# ============================================================

print("\nLoading TEST data...")

test_files = sorted(TEST_DIR.glob("*__horizontal_well.csv"))

test_data = []

for file in tqdm(test_files):

    well_name = file.stem.split("__")[0]

    df = pd.read_csv(file)

    df["WELL"] = well_name
    df["ROW_IDX"] = np.arange(len(df))

    test_data.append(df)

test_df = pd.concat(test_data, ignore_index=True)

print(f"Test Shape: {test_df.shape}")

# ============================================================
# CHECK COLUMNS
# ============================================================

print("\nTrain Columns:")
print(train_df.columns)

print("\nTest Columns:")
print(test_df.columns)

# ============================================================
# BASIC EDA
# ============================================================

print("\nTarget Statistics:")
print(train_df["TVT"].describe())

# ============================================================
# TARGET DISTRIBUTION
# ============================================================

fig = px.histogram(
    train_df,
    x="TVT",
    nbins=100,
    title="TVT Distribution"
)

fig.show()

# ============================================================
# SAMPLE WELL VISUALIZATION
# ============================================================

sample_well = train_df["WELL"].unique()[0]

well_df = train_df[train_df["WELL"] == sample_well]

fig = px.line(
    well_df,
    x="MD",
    y=["GR", "TVT_input"],
    title=f"Sample Well: {sample_well}"
)

fig.show()

# ============================================================
# FEATURE ENGINEERING
# ============================================================

print("\nStarting Feature Engineering...")

# ------------------------------------------------------------
# ROLLING FEATURES
# ------------------------------------------------------------

windows = [5, 10, 20, 50]

for window in windows:

    # GR rolling mean
    train_df[f"GR_roll_mean_{window}"] = (
        train_df.groupby("WELL")["GR"]
        .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

    test_df[f"GR_roll_mean_{window}"] = (
        test_df.groupby("WELL")["GR"]
        .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

    # GR rolling std
    train_df[f"GR_roll_std_{window}"] = (
        train_df.groupby("WELL")["GR"]
        .transform(lambda x: x.rolling(window, min_periods=1).std())
    )

    test_df[f"GR_roll_std_{window}"] = (
        test_df.groupby("WELL")["GR"]
        .transform(lambda x: x.rolling(window, min_periods=1).std())
    )

# ------------------------------------------------------------
# TVT INPUT ROLLING FEATURES
# ------------------------------------------------------------

for window in windows:

    train_df[f"TVT_roll_mean_{window}"] = (
        train_df.groupby("WELL")["TVT_input"]
        .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

    test_df[f"TVT_roll_mean_{window}"] = (
        test_df.groupby("WELL")["TVT_input"]
        .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

# ------------------------------------------------------------
# LAG FEATURES
# ------------------------------------------------------------

lags = [1, 2, 5, 10]

for lag in lags:

    # GR lag
    train_df[f"GR_lag_{lag}"] = (
        train_df.groupby("WELL")["GR"].shift(lag)
    )

    test_df[f"GR_lag_{lag}"] = (
        test_df.groupby("WELL")["GR"].shift(lag)
    )

    # TVT_input lag
    train_df[f"TVT_input_lag_{lag}"] = (
        train_df.groupby("WELL")["TVT_input"].shift(lag)
    )

    test_df[f"TVT_input_lag_{lag}"] = (
        test_df.groupby("WELL")["TVT_input"].shift(lag)
    )

# ------------------------------------------------------------
# GRADIENT FEATURES
# ------------------------------------------------------------

train_df["GR_gradient"] = (
    train_df.groupby("WELL")["GR"].diff()
)

test_df["GR_gradient"] = (
    test_df.groupby("WELL")["GR"].diff()
)

train_df["TVT_gradient"] = (
    train_df.groupby("WELL")["TVT_input"].diff()
)

test_df["TVT_gradient"] = (
    test_df.groupby("WELL")["TVT_input"].diff()
)

# ------------------------------------------------------------
# TRAJECTORY FEATURES
# ------------------------------------------------------------

for df in [train_df, test_df]:

    df["dX"] = df.groupby("WELL")["X"].diff()
    df["dY"] = df.groupby("WELL")["Y"].diff()
    df["dZ"] = df.groupby("WELL")["Z"].diff()

    df["trajectory_distance"] = np.sqrt(
        df["dX"]**2 +
        df["dY"]**2 +
        df["dZ"]**2
    )

# ============================================================
# FILL MISSING VALUES
# ============================================================

train_df = train_df.fillna(method="ffill")
train_df = train_df.fillna(method="bfill")

test_df = test_df.fillna(method="ffill")
test_df = test_df.fillna(method="bfill")

# ============================================================
# COMMON FEATURES ONLY
# ============================================================

common_cols = list(
    set(train_df.columns).intersection(set(test_df.columns))
)

remove_cols = [
    "WELL",
    "TVT",
]

for col in remove_cols:
    if col in common_cols:
        common_cols.remove(col)

features = sorted(common_cols)

print(f"\nTotal Features: {len(features)}")

# ============================================================
# TRAINING DATA
# ============================================================

train_mask = train_df["TVT"].notnull()

X = train_df.loc[train_mask, features]
y = train_df.loc[train_mask, "TVT"]

groups = train_df.loc[train_mask, "WELL"]

X_test = test_df[features]

print(f"\nTrain Matrix Shape: {X.shape}")
print(f"Test Matrix Shape: {X_test.shape}")

# ============================================================
# LIGHTGBM PARAMETERS
# ============================================================

params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 64,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "random_state": 42,
    "n_estimators": 5000,
    "verbosity": -1,
}

# ============================================================
# GROUP KFOLD VALIDATION
# ============================================================

folds = GroupKFold(n_splits=5)

oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

scores = []

print("\nStarting Training...")

for fold, (train_idx, valid_idx) in enumerate(
    folds.split(X, y, groups)
):

    print("\n" + "=" * 50)
    print(f"FOLD {fold + 1}")
    print("=" * 50)

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200)
        ]
    )

    valid_preds = model.predict(X_valid)

    oof[valid_idx] = valid_preds

    fold_rmse = rmse(y_valid, valid_preds)

    scores.append(fold_rmse)

    print(f"Fold RMSE: {fold_rmse:.5f}")

    test_preds += model.predict(X_test) / folds.n_splits

    gc.collect()

# ============================================================
# FINAL SCORE
# ============================================================

print("\n" + "=" * 60)
print(f"CV RMSE: {np.mean(scores):.5f}")
print("=" * 60)

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print("\nTop Features:")
print(importance_df.head(20))

fig = px.bar(
    importance_df.head(30),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 30 Feature Importance"
)

fig.show()

# ============================================================
# CREATE SUBMISSION
# ============================================================

print("\nCreating Submission...")

submission = pd.read_csv(SUBMISSION_PATH)

submission["tvt"] = test_preds[:len(submission)]

submission.to_csv(
    OUTPUT_DIR / "submission.csv",
    index=False
)

print("\nSubmission Saved:")
print(OUTPUT_DIR / "submission.csv")

print("\nSubmission Preview:")
print(submission.head())

# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 60)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 60)

ROGII - Wellbore Geology Prediction
Target: Predict TVT along horizontal wells
Metric: RMSE

Loading TRAIN data...


  0%|          | 0/773 [00:00<?, ?it/s]

Train Shape: (5092255, 15)

Loading TEST data...


  0%|          | 0/3 [00:00<?, ?it/s]

Test Shape: (19221, 8)

Train Columns:
Index(['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA',
       'TVT', 'GR', 'TVT_input', 'WELL', 'ROW_IDX'],
      dtype='object')

Test Columns:
Index(['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input', 'WELL', 'ROW_IDX'], dtype='object')

Target Statistics:
count    5.092255e+06
mean     1.150364e+04
std      6.399711e+02
min      9.245190e+03
25%      1.098793e+04
50%      1.135451e+04
75%      1.203826e+04
max      1.289389e+04
Name: TVT, dtype: float64
